# Predicting arrears next month

**Question:** can we predict `arrears_next_month` from what is known at the end of the billing month?

Design decisions (following `checklist.md`):
- **Leakage:** `final_notice_sent` is recorded in the cycle *after* the billing month, so it is not known at prediction time (and it is nearly a proxy for the label). It is dropped, as is the identifier `meter_ref`. `billing_month` is used only for splitting.
- **Split:** the model will be used on future months and accounts appear in many rows, so the test set is the *latest* months *and* *whole held-out accounts*. Training rows come from earlier months and other accounts only.
- **CV:** inside training rows only, forward-chaining by month blocks, with validation accounts disjoint from fitting accounts.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, brier_score_loss
df = pd.read_csv('data.csv', parse_dates=['billing_month'])
df['price_per_kwh'] = df.bill_eur / df.kwh   # row-local, known at prediction time
df['kwh_per_occupant'] = df.kwh / df.occupants
print(df.shape, 'base rate', df.arrears_next_month.mean().round(3))

## 1. Test set, set aside before any modelling
Test = last 6 billing months (2025-07 onward) restricted to a random 25% of accounts. Training = earlier months (up to 2025-05, leaving a one-month gap) restricted to the other accounts. Rows of test accounts in the past and of training accounts in the future are not used.

In [ ]:
rng = np.random.RandomState(0)
accts = df.meter_ref.unique()
test_accts = set(rng.choice(accts, size=int(0.25*len(accts)), replace=False))
is_test_acct = df.meter_ref.isin(test_accts)
train = df[~is_test_acct & (df.billing_month <= '2025-05-01')].reset_index(drop=True)
test  = df[ is_test_acct & (df.billing_month >= '2025-07-01')].reset_index(drop=True)
assert not set(train.meter_ref) & set(test.meter_ref)
print(len(train), len(test), train.arrears_next_month.mean().round(3), test.arrears_next_month.mean().round(3))

## 2. Candidates, named before scoring
Features (all known at prediction time): `tariff, occupants, kwh, bill_eur, late_payments_12m, price_per_kwh, kwh_per_occupant`.

1. **baseline** - predicts the training prior (trivial).
2. **logreg** - standardised numerics + one-hot tariff, C in {0.01, 0.1, 1, 10}.
3. **hgb** - HistGradientBoosting, (max_depth, learning_rate) in {2,3,None}x{0.03, 0.1}, 200 iterations, min_samples_leaf=100.

Selection metric: mean CV average precision (PR-AUC; the positive class is ~12%). Threshold: maximise F1 on pooled out-of-fold predictions of the selected model.

In [ ]:
num = ['occupants','kwh','bill_eur','late_payments_12m','price_per_kwh','kwh_per_occupant']; cat = ['tariff']
feats = num + cat
def logreg(C): return make_pipeline(ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),cat)]), LogisticRegression(C=C,max_iter=1000))
def hgb(d,lr): return make_pipeline(ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(handle_unknown='ignore'),cat)]),
                                     HistGradientBoostingClassifier(max_depth=d,learning_rate=lr,max_iter=200,min_samples_leaf=100,random_state=0))
candidates = {'baseline': DummyClassifier(strategy='prior')}
for C in [0.01,0.1,1,10]: candidates[f'logreg C={C}'] = logreg(C)
for d in [2,3,None]:
    for lr in [0.03,0.1]: candidates[f'hgb depth={d} lr={lr}'] = hgb(d,lr)
list(candidates)

## 3. Cross-validation inside training rows
Forward-chaining: 4 validation blocks of 3 months at the end of the training period. For each block, fit on rows from earlier months (one-month gap) of accounts *not* in the validation group; validate on the block's rows of a held-out fifth of training accounts. Pipelines are cloned, so all preprocessing is refit inside each fold.

In [ ]:
months = np.sort(train.billing_month.unique())
blocks = [months[-12:-9], months[-9:-6], months[-6:-3], months[-3:]]
tr_accts = train.meter_ref.unique()
rng = np.random.RandomState(1)
grp = pd.Series(rng.randint(0,5,len(tr_accts)), index=tr_accts)   # account -> group
g = train.meter_ref.map(grp)
folds=[]
for k,b in enumerate(blocks):
    val_g = k % 5
    va = np.where(train.billing_month.isin(b) & (g==val_g))[0]
    fit = np.where((train.billing_month < pd.Timestamp(b[0]) - pd.offsets.MonthBegin(1)) & (g!=val_g))[0]
    assert not set(train.meter_ref[va]) & set(train.meter_ref[fit])
    folds.append((fit,va))
print([(len(f),len(v)) for f,v in folds])
y = train.arrears_next_month.values
oof = {}; rows=[]
for name,m in candidates.items():
    p = np.full(len(train), np.nan); aps=[]; aucs=[]
    for fit,va in folds:
        mm = clone(m).fit(train.iloc[fit][feats], y[fit])
        p[va] = mm.predict_proba(train.iloc[va][feats])[:,1]
        aps.append(average_precision_score(y[va],p[va])); aucs.append(roc_auc_score(y[va],p[va]))
    oof[name]=p; rows.append((name,np.mean(aps),np.std(aps),np.mean(aucs)))
cv = pd.DataFrame(rows,columns=['model','cv_AP','AP_sd','cv_AUC']).sort_values('cv_AP',ascending=False); cv

## 4. Choice (CV only)

In [ ]:
best = cv.iloc[0].model
va_idx = np.concatenate([v for _,v in folds]); po = oof[best][va_idx]; yo = y[va_idx]
ths = np.linspace(0.02,0.9,89); f1s=[f1_score(yo,po>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(thr,2), '| CV F1:', round(max(f1s),3))

**Recorded choice (written before the test set is scored):** the model and threshold printed above - the top CV average-precision candidate, with the F1-maximising threshold on pooled out-of-fold predictions. No changes after seeing test results.

## 5. Final fit and single test score

In [ ]:
final = clone(candidates[best]).fit(train[feats], y)
pt = final.predict_proba(test[feats])[:,1]; yt = test.arrears_next_month.values
base = clone(candidates['baseline']).fit(train[feats], y).predict_proba(test[feats])[:,1]
res = pd.DataFrame({'chosen':[roc_auc_score(yt,pt), average_precision_score(yt,pt), brier_score_loss(yt,pt), precision_score(yt,pt>=thr), recall_score(yt,pt>=thr), f1_score(yt,pt>=thr)],
                    'baseline':[0.5, yt.mean(), brier_score_loss(yt,base), np.nan, np.nan, np.nan]},
                   index=['ROC AUC','Average precision','Brier','Precision@thr','Recall@thr','F1@thr'])
print('test rows', len(test), 'positives', yt.sum()); res.round(3)

## Conclusion
Headline (test set, scored once; results from running this notebook's code): chosen logistic regression (C=0.01), ROC AUC ~0.72, average precision ~0.47 vs 0.22 for the base rate, precision ~0.29 / recall ~0.81 at the CV threshold 0.05 (F1 ~0.43). So there is real but moderate signal, mainly from `late_payments_12m`.

**Important drift caveat:** the arrears rate rises steeply over time (about 2-4% in 2023-24, 7% in 2025Q1, 16-25% in 2025Q2-Q4; training rate 5.7% vs test 22%). Predicted probabilities are therefore miscalibrated on the test period and the CV-chosen threshold (0.05) is far too low for it (high recall, low precision). Ranking quality (AUC/AP) is the more trustworthy result; probabilities/threshold would need recalibration on recent data before operational use. Other caveats: `final_notice_sent` was excluded because it is not available at prediction time (including it would give an inflated, unusable score); the test set has a limited number of accounts, so uncertainty is substantial; nothing was tuned after test scoring.